# Lab 3 - Testowanie aplikacji w Pythonie

Zadania zaliczeniowe:
- **Zadanie 1**: Klasa `Product` + testy `unittest` z `setUp` i `assertRaises`
- **Zadanie 2**: Te same testy `Product` w `pytest` (`@pytest.fixture`, `@pytest.mark.parametrize`, `pytest.raises`)

Kod jest podzielony na pliki `.py` (bo `pytest`/`unittest` wymagają plików), a ten notebook prezentuje zawartosc i uruchamia testy.

## Zadanie 1: Klasa `Product`

Klasa z atrybutami `name`, `price`, `quantity` oraz metodami `add_stock`, `remove_stock`, `is_available`, `total_value`.
Walidacja: `price >= 0`, `quantity >= 0`, brak ujemnych `amount` w add/remove.

In [1]:
# Zawartosc pliku product.py
!cat product.py


# -*- coding: utf-8 -*-
"""Klasa Product -- zadanie do samodzielnego wykonania."""


class Product:
    """Reprezentuje produkt w sklepie internetowym."""

    def __init__(self, name: str, price: float, quantity: int):
        if price < 0:
            raise ValueError("price nie moze byc ujemny")
        if quantity < 0:
            raise ValueError("quantity nie moze byc ujemne")
        self.name = name
        self.price = price
        self.quantity = quantity

    def add_stock(self, amount: int):
        """Dodaje okreslona ilosc produktow do magazynu.

        Raises:
            ValueError: jesli amount jest ujemne
        """
        if amount < 0:
            raise ValueError("amount nie moze byc ujemny")
        self.quantity += amount

    def remove_stock(self, amount: int):
        """Usuwa okreslona ilosc produktow z magazynu.

        Raises:
            ValueError: jesli amount jest ujemne lub wieksze niz dostepna ilosc
        """
        if amount < 0:
            

In [2]:
# Demonstracja dzialania klasy
from product import Product

p = Product("Laptop", 2999.99, 10)
print("Stan poczatkowy:", p.name, p.price, p.quantity)
print("Wartosc magazynu:", p.total_value())
print("Dostepny?", p.is_available())

p.add_stock(5)
print("\nPo add_stock(5):", p.quantity)

p.remove_stock(3)
print("Po remove_stock(3):", p.quantity)

# Test walidacji
try:
    Product("X", -1, 0)
except ValueError as e:
    print(f"\nBlad (zgodnie z oczekiwaniem): {e}")

try:
    p.remove_stock(999)
except ValueError as e:
    print(f"Blad (zgodnie z oczekiwaniem): {e}")


Stan poczatkowy: Laptop 2999.99 10
Wartosc magazynu: 29999.899999999998
Dostepny? True

Po add_stock(5): 15
Po remove_stock(3): 12

Blad (zgodnie z oczekiwaniem): price nie moze byc ujemny
Blad (zgodnie z oczekiwaniem): nie mozna usunac wiecej niz jest na stanie


## Zadanie 1: Testy unittest

Testy w `test_product_unittest.py`. Uzywaja:
- `setUp()` zeby tworzyc nowa instancje `Product` przed kazdym testem
- `assertEqual`, `assertTrue`, `assertFalse` dla porownan
- `assertRaises(ValueError)` dla testowania bledow

In [3]:
# Zawartosc pliku test_product_unittest.py
!cat test_product_unittest.py


# -*- coding: utf-8 -*-
"""Testy unittest dla klasy Product -- uzupelnij metody testowe!

Uruchomienie: python -m unittest test_product_unittest -v
"""

import unittest
from product import Product


class TestProduct(unittest.TestCase):

    def setUp(self):
        """Przygotuj instancje Product do testow."""
        self.product = Product("Laptop", 2999.99, 10)

    # --- Testy add_stock ---

    def test_add_stock_positive(self):
        """Sprawdz, czy dodanie towaru zwieksza quantity."""
        self.product.add_stock(5)
        self.assertEqual(self.product.quantity, 15)

    def test_add_stock_negative_raises(self):
        """Sprawdz, czy ujemna wartosc rzuca ValueError."""
        with self.assertRaises(ValueError):
            self.product.add_stock(-5)

    # --- Testy remove_stock ---

    def test_remove_stock_positive(self):
        """Sprawdz, czy usuniecie towaru zmniejsza quantity."""
        self.product.remove_stock(3)
        self.assertEqual(self.product.quantity, 

In [4]:
# Uruchomienie testow unittest
!python -m unittest test_product_unittest -v


test_add_stock_negative_raises (test_product_unittest.TestProduct.test_add_stock_negative_raises)
Sprawdz, czy ujemna wartosc rzuca ValueError. ... ok
test_add_stock_positive (test_product_unittest.TestProduct.test_add_stock_positive)
Sprawdz, czy dodanie towaru zwieksza quantity. ... ok
test_is_available_when_in_stock (test_product_unittest.TestProduct.test_is_available_when_in_stock)
Sprawdz, czy produkt z quantity > 0 jest dostepny. ... ok
test_is_not_available_when_empty (test_product_unittest.TestProduct.test_is_not_available_when_empty)
Sprawdz, czy produkt z quantity == 0 nie jest dostepny. ... ok
test_remove_stock_negative_raises (test_product_unittest.TestProduct.test_remove_stock_negative_raises)
Sprawdz, czy ujemna wartosc rzuca ValueError. ... ok
test_remove_stock_positive (test_product_unittest.TestProduct.test_remove_stock_positive)
Sprawdz, czy usuniecie towaru zmniejsza quantity. ... ok
test_remove_stock_too_much_raises (test_product_unittest.TestProduct.test_remove_sto

## Zadanie 2: Testy pytest

Te same testy przepisane w stylu `pytest`. Uzywaja:
- `@pytest.fixture` zamiast `setUp()`
- `@pytest.mark.parametrize` dla testow z roznymi danymi (jeden test, 3 zestawy dla `add_stock`)
- `pytest.raises(ValueError)` zamiast `assertRaises`

In [5]:
# Zawartosc pliku test_product_pytest.py
!cat test_product_pytest.py


# -*- coding: utf-8 -*-
"""Testy pytest dla klasy Product -- uzupelnij!

Uruchomienie: pytest test_product_pytest.py -v
"""

import pytest
from product import Product


# --- Fixture ---

@pytest.fixture
def product():
    """Tworzy instancje Product do testow (odpowiednik setUp)."""
    return Product("Laptop", 2999.99, 10)


# --- Testy z fixture ---

def test_is_available(product):
    """Sprawdz dostepnosc produktu."""
    assert product.is_available() == True


def test_total_value(product):
    """Sprawdz wartosc calkowita."""
    assert product.total_value() == 2999.99 * 10


# --- Testy z parametryzacja ---

@pytest.mark.parametrize("amount, expected_quantity", [
    (5, 15),     # dodanie 5 do poczatkowych 10 = 15
    (0, 10),     # dodanie 0 = bez zmian
    (100, 110),  # dodanie 100
])
def test_add_stock_parametrized(product, amount, expected_quantity):
    """Testuje add_stock z roznymi wartosciami."""
    product.add_stock(amount)
    assert product.quantity == expected_qu

In [6]:
# Uruchomienie testow pytest
!pytest test_product_pytest.py -v


============================= test session starts ==============================
platform linux -- Python 3.13.2, pytest-9.0.3, pluggy-1.6.0 -- /home/eg/Desktop/zadania/.venv/bin/python3
cachedir: .pytest_cache
rootdir: /home/eg/Desktop/zadania/lab_03
configfile: pytest.ini
plugins: anyio-4.13.0
collected 7 items                                                              

test_product_pytest.py::test_is_available PASSED                         [ 14%]
test_product_pytest.py::test_total_value PASSED                          [ 28%]
test_product_pytest.py::test_add_stock_parametrized[5-15] PASSED         [ 42%]
test_product_pytest.py::test_add_stock_parametrized[0-10] PASSED         [ 57%]
test_product_pytest.py::test_add_stock_parametrized[100-110] PASSED      [ 71%]
test_product_pytest.py::test_remove_stock_too_much_raises PASSED         [ 85%]
test_product_pytest.py::test_add_stock_negative_raises PASSED            [100%]

============================== 7 passed in 0.02s ============

## Pliki w tym folderze

- `product.py` - klasa `Product` (zadanie 1+2 - implementacja)
- `test_product_unittest.py` - testy unittest (zadanie 1)
- `test_product_pytest.py` - testy pytest (zadanie 2)
- `conftest.py` - globalna fixture `calc` (z materialow prowadzacego)
- `pytest.ini` - konfiguracja pytest (z materialow)
- `calculator.py` - klasa referencyjna (z materialow)